# Retrieval Evaluation Query Generation

## 1. Introduction

### 1.1. Install and Import Libraries

In [ ]:
# %pip uninstall -y torchvision
# %pip install --upgrade --force-reinstall --index-url https://download.pytorch.org/whl/cu128 torch==2.11.0+cu128 torchaudio==2.11.0+cu128
# %pip install --upgrade --force-reinstall "transformers==4.51.3" "huggingface_hub==0.30.2" "tokenizers==0.21.1" accelerate sentencepiece protobuf safetensors pandas tqdm

In [1]:
# ============================================
# 1. Standard Library
# ============================================
import os
import json
from pathlib import Path
from importlib import reload

# ============================================
# 2. Third-party Libraries
# ============================================
import pandas as pd
import torch
from tqdm.auto import tqdm

# ============================================
# 3. Project Helpers
# ============================================
import src.retrieval_llms as retrieval_llm_helpers
reload(retrieval_llm_helpers)

from src.retrieval_llms import (
    QUERY_GENERATION_MODEL_ID,
    RELEVANCE_JUDGE_MODEL_ID,
    QueryGenerationLLM,
    RelevanceJudgeLLM,
    extract_first_json_object,
)

In [ ]:
# Dependency sanity check. If this fails after running the install cell,
# restart the notebook kernel and rerun imports before loading the LLMs.
import importlib.util
import transformers
import huggingface_hub
import torch

print(f"transformers: {transformers.__version__}")
print(f"huggingface_hub: {huggingface_hub.__version__}")
print(f"tokenizers installed: {importlib.util.find_spec('tokenizers') is not None}")
print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA runtime: {torch.version.cuda}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}")
print(f"torchvision installed: {importlib.util.find_spec('torchvision') is not None}")

### 1.2. Initialize LLM Services

In [ ]:
# Read the Hugging Face token from Colab userdata when available, then fall back to the environment.
try:
    from google.colab import userdata
    HUGGING_FACE_TOKEN = userdata.get("HUGGING_FACE_TOKEN")
except Exception:
    HUGGING_FACE_TOKEN = os.getenv("HUGGING_FACE_TOKEN")

LLM_LOAD_IN_4BIT = False
LLM_TORCH_DTYPE = torch.bfloat16 if torch.cuda.is_available() else "auto"

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

print(f"Query generation model: {QUERY_GENERATION_MODEL_ID}")
print(f"Relevance judge model: {RELEVANCE_JUDGE_MODEL_ID}")
print(f"Hugging Face token configured: {bool(HUGGING_FACE_TOKEN)}")

Initialize the query-generation LLM. This model is used to generate realistic public-archive retrieval queries and qrels.

In [ ]:
query_generation_llm = QueryGenerationLLM(
    token=HUGGING_FACE_TOKEN,
    torch_dtype=LLM_TORCH_DTYPE,
    load_in_4bit=LLM_LOAD_IN_4BIT,
)

print(f"Loaded query-generation LLM on: {query_generation_llm.device}")

Optional: initialize the separate relevance-judge LLM now.

In [ ]:
INITIALIZE_RELEVANCE_JUDGE_LLM = False

if INITIALIZE_RELEVANCE_JUDGE_LLM:
    relevance_judge_llm = RelevanceJudgeLLM(
        token=HUGGING_FACE_TOKEN,
        torch_dtype=LLM_TORCH_DTYPE,
        load_in_4bit=LLM_LOAD_IN_4BIT,
    )
    print(f"Loaded relevance-judge LLM on: {relevance_judge_llm.device}")
else:
    relevance_judge_llm = None
    print("Relevance judge initialization skipped for now.")

Verify the query-generation LLM returns parseable JSON.

In [ ]:
llm_smoke_test_chunk = {
    "chunk_id": "smoke_test_chunk",
    "dataset": "smoke_test",
    "modality": "text",
    "title": "Archive excerpt",
    "access_level": "public",
    "sensitivity_level": "low",
    "masked_text": (
        "A city council report says the Riverside Public Library opened a digital archive "
        "in 2024 to preserve oral histories, photographs, and meeting records from local residents."
    ),
}

llm_smoke_test_response = query_generation_llm.generate_query_record(
    llm_smoke_test_chunk,
    max_new_tokens=256,
)
print(llm_smoke_test_response)

llm_smoke_test_record = extract_first_json_object(llm_smoke_test_response)
display(pd.DataFrame([llm_smoke_test_record]))